# Step 2: Identify and Filter Major Automakers

## Purpose
From the broad person–application table produced in Step 1, identify all
entities with more than 400 total US patent applications (202 psn_names), then
retain only the 90 *major* automotive manufacturers manually selected from that
pool by Andrey Gusak based on open data from Internet resources — excluding
entities unrelated to automotive manufacturing.

## Input
| File | Description |
|------|-------------|
| `../step1/maker_app_links.csv` | All firm–application links from Step 1 |

## Outputs
| File | Description |
|------|-------------|
| `makers_above_400.csv` | All 202 psn_names with >400 applications (pre-selection pool) |
| `major_makers.csv` | Filtered to 90 major automakers — read by **Step 4** |
| `patent_counts_by_maker.xlsx` | Ranked application counts (inspection aid) |


In [ ]:
import pandas as pd
from pathlib import Path


In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
INPUT_FILE        = '../step1/maker_app_links.csv'
OUTPUT_CSV        = 'major_makers.csv'
OUTPUT_XLSX       = 'patent_counts_by_maker.xlsx'
OUTPUT_ABOVE_400  = 'makers_above_400.csv'

# Minimum number of applications a firm must have to enter the candidate pool.
# This threshold (400) was set to exclude very small filers whose patent
# activity is too sparse for reliable time-series analysis.
# Applying it yields 202 candidate psn_names.
MIN_APPLICATIONS = 400

# ── Curated list of 90 major automakers ──────────────────────────────────────
# These 90 psn_names were manually pre-selected by Andrey Gusak from the 202
# entities that exceed the MIN_APPLICATIONS threshold, based on open data from
# Internet resources.  Entities unrelated to automotive manufacturing (e.g.
# aerospace, electronics, or general industrial firms) were excluded.
# The resulting set is saved as makers_above_400.csv for transparency.
#
# Firms are grouped by country of origin for transparency.
# The German group (VW, Daimler, BMW and their sub-brands) is the *treatment*
# group in the DDD regression; all others are the *control* group.

GERMAN_MAKERS = [
    'ADAM OPEL', 'ADAM OPEL AG, 6090 RUESSELSHEIM',
    'ADAM OPEL AG, 65428 RUESSELSHEIM',
    'AUDI', 'BMW (BAYERISCHE MOTOREN WERKE)',
    'DAIMLER', 'DAIMLER-BENZ', 'DAIMLERCHRYSLER', 'MERCEDES-BENZ',
    'MERCEDES-BENZ AKTIENGESELLSCHAFT, 7000 STUTTGART',
    'MERCEDES-BENZ AKTIENGESELLSCHAFT, 70327 STUTTGART',
    'PORSCHE', 'VOLKSWAGEN',
]

JAPANESE_MAKERS = [
    'HONDA ACCESS CORPORATION', 'HONDA ELECTRONIC COMPANY', 'HONDA LOCK',
    'HONDA LOCK MANUFACTURING COMPANY', 'HONDA MOTOR COMPANY',
    'HONDA TSUSHIN KOGYO COMPANY', 'MAZDA MOTOR CORPORATION',
    'MITSUBISHI AUTOMOB ENG COMPANY',
    'MITSUBISHI FUSO TRUCK & BUS CORPORATION',
    'MITSUBISHI FUSO TRUCK AND BUS CORPORATION',
    'MITSUBISHI HEAVY INDUSTRIES AUTOMOTIVE THERMAL SYSTEMS COMPANY',
    'MITSUBISHI HEAVY INDUSTRIES ENGINE & TURBOCHARGER',
    'MITSUBISHI HEAVY INDUSTRIES ENGINEERING',
    'MITSUBISHI HEAVY INDUSTRIES THERMAL SYSTEMS COMPANY',
    'MITSUBISHI JIDOSHA ENGINEERING', 'MITSUBISHI MOTORS CORPORATION',
    'NISSAN MOTOR COMPANY', 'NISSAN NORTH AMERICA', 'NISSAN SHATAI COMPANY',
    'SUBARU', 'SUZUKI JIDOSHA KOGYO', 'SUZUKI MOTOR CORPORATION',
    'TOYOTA AUTO BODY COMPANY', 'TOYOTA BOSHOKU CORPORATION',
    'TOYOTA CENTRAL RES & DEV LAB', 'TOYOTA CHUO KENKYUSHO', 'TOYOTA HOME',
    'TOYOTA INDUSTRIES CORPORATION', 'TOYOTA INFOTECHNOLOGY CENTER COMPANY',
    'TOYOTA MOTOR CORPORATION',
    'TOYOTA MOTOR EAST JAPAN',
    'TOYOTA MOTOR ENGINEERING & MANUFACTURING NORTH AMERICA',
    'TOYOTA MOTOR EUROPE', 'TOYOTA SHATAI', 'TOYOTA TSUSHO CORPORATION',
]

KOREAN_MAKERS = [
    'HYUNDAI AUTONET COMPANY', 'HYUNDAI AUTRON COMPANY', 'HYUNDAI DYMOS',
    'HYUNDAI KEFICO CORPORATION', 'HYUNDAI MOBIS COMPANY',
    'HYUNDAI MOTOR COMPANY', 'HYUNDAI TRANSYS', 'HYUNDAI WIA CORPORATION',
    'KIA CORPORATION', 'KIA MOTORS CORPORATION',
]

AMERICAN_MAKERS = [
    'FORD GLOBAL TECHNOLOGIES', 'FORD MOTOR COMPANY', 'FORD-WERKE',
    'GMC', 'GM (GENERAL MOTORS CORPORATION)', 'GM GLOBAL TECHNOLOGY OPERATIONS' ,'LINCOLN', 'LINCOLN GLOBAL',
    'SAIC GENERAL MOTORS CORPORATION', 'SAIC-GM-WULING AUTOMOBILE COMPANY' ,'SHANGHAI GENERAL MOTORS COMPANY',
    'GENERAL MOTORS RESEARCH CORPORATION'
]

SWEDISH_BRITISH_MAKERS = [
    'AB VOLVO', 'AB VOLVO PENTA',
    'JAGUAR CARS', 'JAGUAR LAND ROVER',
    'SAAB', 'SAAB AUTOMOBILE', 'SAAB-SCANIA',
    'VOLVO', 'VOLVO CAR CORPORATION', 'VOLVO LASTVAGNAR',
    'VOLVO TECHNOLOGY CORPORATION', 'VOLVO TRUCK CORPORATION',
]

ITALIAN_MAKERS = [
    'ALFA ROMEO', 'CRF (CENTRO RICERCHE FIAT)', 'FIAT', 'FIAT AUTO',
    'FIAT SOCIETA ANONIMA', 'IVECO FIAT',
]

OTHER_MAKERS = [
    'TESLA', 'TESLA MOTORS',
]

# Combined list of all selected PSN_NAMEs
SELECTED_MAKERS = (
    GERMAN_MAKERS + JAPANESE_MAKERS + KOREAN_MAKERS +
    AMERICAN_MAKERS + SWEDISH_BRITISH_MAKERS + ITALIAN_MAKERS + OTHER_MAKERS
)


In [ ]:
def load_maker_links(filepath: str) -> pd.DataFrame:
    """
    Load the maker–application link table produced by Step 1.

    Parameters
    ----------
    filepath : str
        Relative path to maker_app_links.csv (Step 1 output).

    Returns
    -------
    pd.DataFrame : (person_id, appln_id, psn_name, person_ctry_code)
    """
    df = pd.read_csv(filepath)
    print(f'Loaded {len(df):,} rows — {df["psn_name"].nunique()} unique psn_name values')
    return df


In [ ]:
def count_applications_per_maker(df: pd.DataFrame) -> pd.DataFrame:
    """
    Count distinct patent applications per standardised firm name.

    We count *unique* appln_id values rather than rows because the same
    application can appear multiple times in TLS207 if a single firm has
    multiple person_id entries (e.g. different address variants in PATSTAT).

    Parameters
    ----------
    df : pd.DataFrame — full maker–application links table.

    Returns
    -------
    pd.DataFrame : (psn_name, application_count, share_pct), sorted descending.
    """
    counts = (
        df.groupby('psn_name')['appln_id']
        .nunique()
        .reset_index(name='application_count')
        .sort_values('application_count', ascending=False)
    )
    total = df['appln_id'].nunique()
    counts['share_pct'] = (counts['application_count'] / total * 100).round(2)
    return counts


In [ ]:
def filter_above_threshold(counts: pd.DataFrame,
                            min_apps: int,
                            output_path: str) -> pd.DataFrame:
    """
    Identify all psn_names that exceed the minimum application threshold.

    This produces the candidate pool (202 psn_names) from which the 90 major
    automakers were manually selected by Andrey Gusak.  The full list is saved
    as a CSV for documentation and reproducibility.

    Parameters
    ----------
    counts      : pd.DataFrame — output of count_applications_per_maker().
    min_apps    : int — minimum application count (exclusive lower bound).
    output_path : str — path to save the filtered list.

    Returns
    -------
    pd.DataFrame : (psn_name, application_count) for entities above threshold.
    """
    above = counts[counts['application_count'] > min_apps][['psn_name', 'application_count']].copy()
    print(f'Entities with > {min_apps} applications: {len(above):,} psn_names')
    above.to_csv(output_path, index=False)
    print(f'Saved candidate pool → {output_path}')
    return above


In [ ]:
def filter_to_selected_makers(df: pd.DataFrame,
                               selected_makers: list) -> pd.DataFrame:
    """
    Retain only rows whose psn_name appears in the curated maker list.

    The 90 psn_names in SELECTED_MAKERS were manually chosen by Andrey Gusak
    from the 202-entity candidate pool (all psn_names with > 400 applications),
    based on open data from Internet resources.  Entities unrelated to
    automotive manufacturing were excluded.

    Parameters
    ----------
    df              : pd.DataFrame — full maker–application table.
    selected_makers : list of str — curated PSN_NAME values.

    Returns
    -------
    pd.DataFrame : filtered to 88 major automakers.
    """
    filtered = df[df['psn_name'].isin(selected_makers)].copy()
    print(f'Retained {filtered["psn_name"].nunique()} unique makers, {len(filtered):,} rows')
    return filtered


In [ ]:
def main() -> None:
    """
    Full Step 2 pipeline:
      1. Load Step 1 output.
      2. Count applications per maker and save the ranked list (xlsx).
      3. Identify and save the 202 candidate psn_names with > 400 applications.
      4. Filter to the 90 manually curated major automakers.
      5. Save filtered table as major_makers.csv.
    """
    print('=== Step 2: Filter to Major Automakers ===')

    # Load
    df = load_maker_links(INPUT_FILE)

    # Rank all makers by application count and save for inspection
    counts = count_applications_per_maker(df)
    counts.to_excel(OUTPUT_XLSX, index=False)
    print(f'Saved ranked counts → {OUTPUT_XLSX}')

    # Identify and save all entities above the 400-application threshold.
    # This is the 202-entity candidate pool from which the 90 automakers
    # were manually selected by Andrey Gusak.
    filter_above_threshold(counts, MIN_APPLICATIONS, OUTPUT_ABOVE_400)

    # Filter to the 90 manually curated automakers
    df_major = filter_to_selected_makers(df, SELECTED_MAKERS)

    # Save
    df_major.to_csv(OUTPUT_CSV, index=False)
    print(f'\n✓ Saved → {OUTPUT_CSV}  ({len(df_major):,} rows)')


main()
